# Deploy SageMaker Endpoint with DataCapture & Monitoring Pipeline

In this lab, you will deploy a Small Language Model (SLM) on a SageMaker AI real-time endpoint with **DataCapture** enabled, and learn about the automated monitoring pipeline that processes captured inference data.

**What you will learn:**
- Deploy a Small Language Model (Qwen3-8B) on a SageMaker AI real-time endpoint using JumpStart
- Enable DataCapture on the endpoint to log all request/response pairs to S3
- Verify DataCapture files are being written to S3
- Understand the automated monitoring pipeline architecture (EventBridge → Step Functions → Lambda)
- Verify that the pipeline is logging traces and evaluation scores to MLflow

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
!pip install -U sagemaker==2.253.1 mlflow==3.9.0 --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

## 1. Setup Environment

We start by importing the required libraries and discovering the Studio S3 bucket created by the workshop CloudFormation stack.

In [ ]:
import json
import boto3
import sagemaker
from sagemaker.jumpstart.model import JumpStartModel
from sagemaker.model_monitor import DataCaptureConfig

role = sagemaker.get_execution_role()
sess = sagemaker.Session()

model_id = "huggingface-reasoning-qwen3-8b"
instance_type = "ml.g5.2xlarge"
endpoint_name = "workshop-qwen3-8b"

print(f"Model: {model_id}")
print(f"Instance: {instance_type}")
print(f"Endpoint: {endpoint_name}")

In [ ]:
# Discover the Studio S3 bucket created by CloudFormation
# DataCapture must write to this bucket so that EventBridge can detect new .jsonl files
# and trigger the monitoring pipeline.
s3_client = boto3.client('s3')
account_id = boto3.client('sts').get_caller_identity()['Account']
studio_bucket = None
for b in s3_client.list_buckets()['Buckets']:
    if b['Name'].startswith(f'sagemaker-studio-{account_id}'):
        studio_bucket = b['Name']
        break
assert studio_bucket, 'Studio S3 bucket not found. Ensure the CloudFormation stack deployed successfully.'
print(f'Studio S3 bucket: {studio_bucket}')

## 2. Configure DataCapture

DataCapture records every request and response sent to the endpoint as `.jsonl` files in S3. This enables downstream automated monitoring and evaluation pipelines.

<div style="padding: 15px; background-color: #d1ecf1; border-left: 5px solid #0c5460; color: #0c5460;">
<strong>ℹ️ Note:</strong> DataCapture must write to the Studio S3 bucket (not the default SageMaker bucket) because the automated monitoring pipeline's EventBridge rule watches this specific bucket for new <code>.jsonl</code> files.
</div>

In [ ]:
data_capture_s3_key = f"data-capture/{endpoint_name}"

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f"s3://{studio_bucket}/{data_capture_s3_key}",
    capture_options=["REQUEST", "RESPONSE"],
    json_content_types=["application/json"],
)

print(f"DataCapture destination: {data_capture_config.destination_s3_uri}")

## 3. Deploy the Model with DataCapture Enabled

We deploy **Qwen3-8B** using SageMaker JumpStart. This is an 8-billion parameter reasoning model that supports function calling — which is required for agent tool use in later labs.

> **Note:** Endpoint deployment takes approximately 10-15 minutes. The model requires a GPU instance (ml.g5.2xlarge).

In [ ]:
model = JumpStartModel(
    model_id=model_id,
    role=role,
    sagemaker_session=sess,
    env={
        "OPTION_TENSOR_PARALLEL_DEGREE": "1",
        "OPTION_MAX_MODEL_LEN": "12288",
        "OPTION_GPU_MEMORY_UTILIZATION": "0.95",
        "OPTION_ENABLE_AUTO_TOOL_CHOICE": "true",
        "OPTION_TOOL_CALL_PARSER": "hermes",
    },
)

print(f"Deploying {model_id} on {instance_type}...")
print("This will take approximately 10-15 minutes.\n")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    endpoint_name=endpoint_name,
    accept_eula=True,
    data_capture_config=data_capture_config,
)

print(f"\nEndpoint '{endpoint_name}' is now InService with DataCapture enabled!")

## 4. Verify the Endpoint

Let's send a simple prompt to verify the endpoint is working.

In [ ]:
payload = {
    "messages": [{"role": "user", "content": "You are a medical triage assistant. Assess the following patient and provide the likely condition, triage level, and recommended treatment protocol.\n\nPatient ID: PT-001\nAge: 62\nSymptoms: chest pain, shortness of breath, sweating\nReported Severity: high /no_think"}],
    "parameters": {
        "temperature": 0,
        "top_p": 0.9,
        "max_new_tokens": 512
    }
}

response = predictor.predict(payload)
print(response["choices"][0]["message"]["content"])

## 5. Verify DataCapture Files in S3

Since we enabled DataCapture, the request and response from the test call above should now be captured as a `.jsonl` file in S3.

> **Note:** It may take 1-2 minutes after the endpoint call for the DataCapture files to appear in S3. If you don't see any files, wait a moment and re-run the cell below.

In [ ]:
import time

print(f"Checking for DataCapture files in s3://{studio_bucket}/{data_capture_s3_key}/...")
print("Waiting 60 seconds for DataCapture files to land...")
time.sleep(60)

response = s3_client.list_objects_v2(
    Bucket=studio_bucket,
    Prefix=data_capture_s3_key,
    MaxKeys=50,
)

if "Contents" in response:
    jsonl_files = [obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".jsonl")]
    print(f"\nFound {len(jsonl_files)} DataCapture .jsonl file(s):")
    for f in jsonl_files:
        print(f"  s3://{studio_bucket}/{f}")
else:
    print("\nNo DataCapture files found yet. Wait a moment and re-run this cell.")

## 6. Automated DataCapture Monitoring Pipeline

The workshop CloudFormation stack has deployed an automated monitoring pipeline that processes these DataCapture files. Here's how it works:

```
SageMaker Endpoint (DataCapture) → S3 (.jsonl files) → EventBridge → 
Step Functions → Docker Lambda → MLflow Traces + GenAI Evaluations
```

### Pipeline Components

1. **SageMaker DataCapture** writes `.jsonl` files to S3 for every inference request
2. **Amazon EventBridge** detects new `.jsonl` files via S3 event notifications
3. **AWS Step Functions** orchestrates the processing workflow with retry logic
4. **AWS Lambda** (Docker-based with full MLflow SDK) reads the `.jsonl` file, filters out intermediate tool-call rounds, parses request/response pairs, logs them as MLflow traces, and runs GenAI evaluations

### Automated Evaluation Scorers

The Lambda function runs the following evaluations on each captured inference using Amazon Bedrock as the LLM judge:

| Scorer | Type | Description |
|--------|------|-------------|
| **Safety** | Built-in | Checks if the response is free of harmful content |
| **RelevanceToQuery** | Built-in | Evaluates if the response is relevant to the input |
| **Fluency** | Built-in | Assesses the linguistic quality of the response |
| **follows_objective** | Guidelines | Checks if the response follows the request objective |
| **professional_tone** | Guidelines | Verifies the response maintains a professional tone |
| **coherence** | Custom Judge | Evaluates logical flow and consistency |
| **tokens_words** | Custom Scorer | Counts approximate words in the response |

### Infrastructure Details

The pipeline is deployed via CloudFormation using CodeBuild to build a Docker Lambda image:

- **ECR Repository**: Stores the Docker image with MLflow SDK and evaluation libraries
- **CodeBuild Project**: Builds the Docker image from the workshop GitHub repository during stack creation
- **Lambda Function**: Docker-based (3GB memory, 15min timeout) with mlflow==3.9.0
- **Step Functions State Machine**: Orchestrates Lambda invocation with retry logic
- **EventBridge Rule**: Triggers on new `.jsonl` files in the Studio S3 bucket
- **S3 EventBridge Enabler**: Custom resource that enables EventBridge notifications on the S3 bucket

> **Note:** You do not need to deploy or configure any of these components — they are all pre-deployed as part of the workshop setup. This section is for reference so you understand how the automated monitoring works.

## 7. Verify the Monitoring Pipeline

The test call we made earlier should have triggered the monitoring pipeline. Let's verify by checking MLflow for traces logged by the pipeline.

The pipeline logs traces to the `medical-triage-datacapture` experiment with evaluation scores attached.

In [ ]:
# Retrieve MLflow tracking URI from previous lab
%store -r

import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# The DataCapture pipeline logs to a separate experiment
datacapture_experiment_name = "medical-triage-datacapture"
mlflow.set_experiment(datacapture_experiment_name)

print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"DataCapture Experiment: {datacapture_experiment_name}")
print(f"\nWaiting for the pipeline to process the DataCapture file...")
print(f"This may take 1-2 minutes after the .jsonl file lands in S3.")

In [ ]:
# Check for traces in the datacapture experiment
traces = mlflow.search_traces()

if len(traces) > 0:
    print(f"Found {len(traces)} trace(s) in the '{datacapture_experiment_name}' experiment.")
    print(f"\nThe monitoring pipeline is working! Traces are being logged with evaluation scores.")
    print(f"\nYou can view these traces in the MLflow UI:")
    print(f"  1. Open your SageMaker AI MLflow App")
    print(f"  2. Navigate to the '{datacapture_experiment_name}' experiment")
    print(f"  3. Click the Traces tab to see captured traces with evaluation scores")
else:
    print(f"No traces found yet in '{datacapture_experiment_name}'.")
    print(f"The pipeline may still be processing. Wait 1-2 minutes and re-run this cell.")
    print(f"\nTroubleshooting:")
    print(f"  - Check Step Functions execution history in the AWS Console")
    print(f"  - Check the Lambda function CloudWatch logs for errors")

## 8. Store Variables for Next Labs

We store key variables so they can be retrieved in subsequent notebooks.

In [ ]:
# Store variables for use in subsequent notebooks
%store studio_bucket
%store endpoint_name
%store data_capture_s3_key
%store datacapture_experiment_name

print("Variables stored for next labs:")
print(f"  studio_bucket = {studio_bucket}")
print(f"  endpoint_name = {endpoint_name}")
print(f"  data_capture_s3_key = {data_capture_s3_key}")
print(f"  datacapture_experiment_name = {datacapture_experiment_name}")

## Next Steps

Your endpoint is deployed with DataCapture enabled, and the automated monitoring pipeline is processing inference data in the background.

In the next lab, you will dive deep into **MLflow's evaluation library** — learning about the different types of scorers (built-in, guideline-based, template-based, third-party) and how to use `mlflow.genai.evaluate()` to assess model quality.

> **Important:** Do not delete the endpoint — it will be used in the next labs.

## Cleanup (End of Workshop Only)

When you are done with the entire workshop, uncomment and run the cell below to delete the endpoint.

In [ ]:
# Uncomment the lines below to delete the endpoint and model
# predictor.delete_model()
# predictor.delete_endpoint()
# print(f"Endpoint '{endpoint_name}' deleted successfully.")